# 🔍 Image Forgery Detection — Dataset Exploration

**Deep Learning Based Image Forgery Detection Using Convolutional Neural Networks**

This notebook explores the combined CASIA 2.0 + Columbia dataset to understand:
- Class distribution
- Image dimensions & formats
- Sample visualizations
- Data quality checks

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import random

# Add src to path
sys.path.insert(0, os.path.join('..', 'src'))
from dataset_loader import collect_image_paths, get_train_transforms, get_val_transforms, ForgeryDataset

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Dataset root
DATASET_ROOT = os.path.join('..', 'dataset')
print(f'Dataset root: {os.path.abspath(DATASET_ROOT)}')

## 1. Load Dataset Paths

In [ ]:
image_paths, labels = collect_image_paths(DATASET_ROOT)
print(f'Total images: {len(image_paths)}')
print(f'Authentic (0): {labels.count(0)}')
print(f'Forged (1): {labels.count(1)}')

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = Counter(labels)
class_names = ['Authentic', 'Forged']
colors = ['#4ECDC4', '#FF6B6B']

axes[0].bar(class_names, [counts[0], counts[1]], color=colors, edgecolor='white', linewidth=2)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Images')
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie([counts[0], counts[1]], labels=class_names, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Image Format Distribution

In [ ]:
extensions = [os.path.splitext(p)[1].lower() for p in image_paths]
ext_counts = Counter(extensions)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(ext_counts.keys(), ext_counts.values(), color='#45B7D1', edgecolor='white')
ax.set_title('Image Format Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 50,
            f'{int(height)}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print('Formats:', dict(ext_counts))

## 4. Image Dimension Analysis

In [ ]:
# Sample 200 images for dimension analysis
sample_indices = random.sample(range(len(image_paths)), min(200, len(image_paths)))
widths, heights = [], []

for idx in sample_indices:
    try:
        img = Image.open(image_paths[idx])
        w, h = img.size
        widths.append(w)
        heights.append(h)
    except:
        pass

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(widths, bins=30, color='#FF6B6B', alpha=0.8, edgecolor='white')
axes[0].set_title('Width Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Pixels')

axes[1].hist(heights, bins=30, color='#4ECDC4', alpha=0.8, edgecolor='white')
axes[1].set_title('Height Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Pixels')

plt.tight_layout()
plt.show()
print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

## 5. Sample Images Visualization

In [ ]:
# Show 5 authentic and 5 forged samples
auth_indices = [i for i, l in enumerate(labels) if l == 0]
forg_indices = [i for i, l in enumerate(labels) if l == 1]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Sample Images from Dataset', fontsize=16, fontweight='bold')

# Authentic samples
for i, idx in enumerate(random.sample(auth_indices, 5)):
    img = Image.open(image_paths[idx]).convert('RGB')
    axes[0][i].imshow(img)
    axes[0][i].set_title('Authentic', color='#4ECDC4', fontweight='bold')
    axes[0][i].axis('off')

# Forged samples
for i, idx in enumerate(random.sample(forg_indices, 5)):
    img = Image.open(image_paths[idx]).convert('RGB')
    axes[1][i].imshow(img)
    axes[1][i].set_title('Forged', color='#FF6B6B', fontweight='bold')
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig('../outputs/plots/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Dataset Source Breakdown

In [ ]:
casia_auth = sum(1 for p in image_paths if 'casia' in p.lower() and labels[image_paths.index(p)] == 0)
casia_forg = sum(1 for p in image_paths if 'casia' in p.lower() and labels[image_paths.index(p)] == 1)
col_auth = sum(1 for p in image_paths if 'columbia' in p.lower() and labels[image_paths.index(p)] == 0)
col_forg = sum(1 for p in image_paths if 'columbia' in p.lower() and labels[image_paths.index(p)] == 1)

data = {
    'Dataset': ['CASIA 2.0', 'CASIA 2.0', 'Columbia', 'Columbia'],
    'Class': ['Authentic', 'Forged', 'Authentic', 'Forged'],
    'Count': [casia_auth, casia_forg, col_auth, col_forg]
}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(2)
width = 0.35
ax.bar(x - width/2, [casia_auth, col_auth], width, label='Authentic', color='#4ECDC4')
ax.bar(x + width/2, [casia_forg, col_forg], width, label='Forged', color='#FF6B6B')
ax.set_xticks(x)
ax.set_xticklabels(['CASIA 2.0', 'Columbia'])
ax.set_title('Images per Dataset & Class', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'CASIA 2.0 — Authentic: {casia_auth}, Forged: {casia_forg}')
print(f'Columbia  — Authentic: {col_auth}, Forged: {col_forg}')

## 7. Test DataLoader

In [ ]:
from dataset_loader import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(
    dataset_root=DATASET_ROOT,
    batch_size=8,
    num_workers=0
)

images, labels_batch = next(iter(train_loader))
print(f'Batch shape: {images.shape}')
print(f'Labels: {labels_batch.tolist()}')
print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

## 8. Visualize Augmented Samples

In [ ]:
from utils import denormalize

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Augmented Training Samples', fontsize=16, fontweight='bold')

for i in range(8):
    ax = axes[i // 4][i % 4]
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    label_name = 'Authentic' if labels_batch[i].item() == 0 else 'Forged'
    color = '#4ECDC4' if labels_batch[i].item() == 0 else '#FF6B6B'
    ax.set_title(label_name, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## ✅ Summary

The dataset is ready for training:
- Combined CASIA 2.0 + Columbia images
- Binary labels: 0 (Authentic) vs 1 (Forged)
- Images resized to 224×224 with ImageNet normalization
- Training augmentations include random flip, rotation, and color jitter
- Data split: 70% train / 15% val / 15% test